In [21]:
import cv2
from pynq.lib.video import *
from pynq.lib import AxiGPIO
import numpy as np
import time
from pynq import Overlay, MMIO, allocate

ol = Overlay("./AES_SYS.bit", download = True)
print(ol.ip_dict.keys())

NUM_CHUNKS = 12
PIXELS_PER_CHUNK = 320 * 240
IMG_WIDTH          = 1280
IMG_HEIGHT         = 720
BYTES_PER_PIXEL    = 4

FULL_IMAGE_PIXELS  = IMG_WIDTH         * IMG_HEIGHT
FULL_IMAGE_BYTES   = FULL_IMAGE_PIXELS * BYTES_PER_PIXEL
PIXELS_PER_CHUNK = int(FULL_IMAGE_PIXELS / NUM_CHUNKS)



dict_keys(['axi_gpio_0', 'axi_gpio_1', 'axi_gpio_2', 'axi_gpio_3', 'axi_gpio_4', 'axi_gpio_5', 'axi_gpio_6', 'axi_gpio_7', 'axi_gpio_8', 'axi_gpio_9', 'axi_gpio_10', 'axi_intc_0', 'axi_vdma_0', 'axi_cdma_0', 'processing_system7_0'])


In [22]:
GPIO0_ADDR = ol.ip_dict['axi_gpio_0']['phys_addr']
GPIO0_ADDR_range = ol.ip_dict['axi_gpio_0']['addr_range']
GPIO1_ADDR = ol.ip_dict['axi_gpio_1']['phys_addr']
GPIO1_ADDR_range = ol.ip_dict['axi_gpio_1']['addr_range']
GPIO2_ADDR = ol.ip_dict['axi_gpio_2']['phys_addr']
GPIO2_ADDR_range = ol.ip_dict['axi_gpio_2']['addr_range']
GPIO3_ADDR = ol.ip_dict['axi_gpio_3']['phys_addr']
GPIO3_ADDR_range = ol.ip_dict['axi_gpio_3']['addr_range']
GPIO4_ADDR = ol.ip_dict['axi_gpio_4']['phys_addr']
GPIO4_ADDR_range = ol.ip_dict['axi_gpio_4']['addr_range']
GPIO5_ADDR = ol.ip_dict['axi_gpio_5']['phys_addr']
GPIO5_ADDR_range = ol.ip_dict['axi_gpio_5']['addr_range']
GPIO6_ADDR = ol.ip_dict['axi_gpio_6']['phys_addr']
GPIO6_ADDR_range = ol.ip_dict['axi_gpio_6']['addr_range']
GPIO7_ADDR = ol.ip_dict['axi_gpio_7']['phys_addr']
GPIO7_ADDR_range = ol.ip_dict['axi_gpio_7']['addr_range']
GPIO8_ADDR = ol.ip_dict['axi_gpio_8']['phys_addr']
GPIO8_ADDR_range = ol.ip_dict['axi_gpio_8']['addr_range']
GPIO9_ADDR = ol.ip_dict['axi_gpio_9']['phys_addr']
GPIO9_ADDR_range = ol.ip_dict['axi_gpio_9']['addr_range']

CDMA_ADDR = ol.ip_dict['axi_cdma_0']['phys_addr']
CDMA_ADDR_range = ol.ip_dict['axi_cdma_0']['addr_range']
BRAM0_ADDR = 0xC0000000

In [23]:
KEY_0 = MMIO(GPIO1_ADDR, GPIO1_ADDR_range)
KEY_1 = MMIO(GPIO2_ADDR, GPIO2_ADDR_range)
KEY_2 = MMIO(GPIO3_ADDR, GPIO3_ADDR_range)
KEY_3 = MMIO(GPIO4_ADDR, GPIO4_ADDR_range)
START = MMIO(GPIO5_ADDR, GPIO5_ADDR_range)
ENC_DEC = MMIO(GPIO6_ADDR, GPIO6_ADDR_range)
KEY_GEN_DONE = MMIO(GPIO7_ADDR, GPIO7_ADDR_range)
DONE = MMIO(GPIO8_ADDR, GPIO8_ADDR_range)
KEY_START = MMIO(GPIO9_ADDR, GPIO9_ADDR_range)
cdma = MMIO(CDMA_ADDR, CDMA_ADDR_range)
sw_1 = AxiGPIO(ol.ip_dict["axi_gpio_10"]).channel1
sw_2 = AxiGPIO(ol.ip_dict["axi_gpio_10"]).channel2
vdma = ol.axi_vdma_0
mode = VideoMode(IMG_WIDTH, IMG_HEIGHT, 24)
vdma.writechannel.mode = mode

In [24]:
# KEY
KEY_0.write(0x0,0x09cf4f3c) #LSB
KEY_1.write(0x0,0xabf71588)
KEY_2.write(0x0,0x28aed2a6)
KEY_3.write(0x0,0x2b7e1516) #MSB

KEY_START.write(0x0, 0x1)
while (KEY_GEN_DONE.read(0x0) & 0x1) == 0:
        pass
KEY_START.write(0x0, 0x0)

In [25]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("無法開啟攝影機")
    exit()
else:
    print("攝影機成功開啟")
    
cap.set(cv2.CAP_PROP_FRAME_WIDTH, IMG_WIDTH);
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, IMG_HEIGHT);

buttons_instance = ol.ip_dict["axi_gpio_0"]
buttons = AxiGPIO(buttons_instance).channel1

input_buffer = allocate(shape=(FULL_IMAGE_PIXELS,), dtype=np.uint32)
enc_buffer = allocate(shape=(FULL_IMAGE_PIXELS,), dtype=np.uint32)
dec_buffer = allocate(shape=(FULL_IMAGE_PIXELS,), dtype=np.uint32)

original = allocate(shape=(IMG_HEIGHT, IMG_WIDTH,3), dtype=np.uint8)
enc = allocate(shape=(IMG_HEIGHT, IMG_WIDTH, 3), dtype=np.uint8)
dec = allocate(shape=(IMG_HEIGHT, IMG_WIDTH, 3), dtype=np.uint8)

vdma.writechannel.start()

while True:
    time.sleep(0.1)
    ret, frame_cap = cap.read()
    b, g, r = cv2.split(frame_cap)
    img = cv2.merge([r,g,b]) 
    packed = (r << 16) | (g << 8) | b
    flatten = packed.flatten()
    np.copyto(input_buffer, flatten)

    for i in range(12):
        cdma.write(0x00, 0x4)  # Reset
        cdma.write(0x18, int(input_buffer.physical_address + i * PIXELS_PER_CHUNK * 4))  # Source from HP port
        cdma.write(0x20, BRAM0_ADDR)  # Destination to BRAM
        cdma.write(0x28, int(PIXELS_PER_CHUNK * 4))  # Total byte count
        cdma.write(0x00, 0x1)  # Start transfer

        while (cdma.read(0x04) & 0x2) == 0:
            pass

        ENC_DEC.write(0x0, 0x0)
        START.write(0x0, 0x1)

        while (DONE.read(0x0) & 0x1) == 0:
            pass
        START.write(0x0, 0x0)

        cdma.write(0x00, 0x4)  # Reset
        cdma.write(0x18, BRAM0_ADDR)  # Source from HP port
        cdma.write(0x20, int(enc_buffer.physical_address + i * PIXELS_PER_CHUNK * 4))  # Destination to BRAM
        cdma.write(0x28, int(PIXELS_PER_CHUNK * 4))  # Total byte count
        cdma.write(0x00, 0x1)  # Start transfer
        while (cdma.read(0x04) & 0x2) == 0:
            pass 

        #cdma.write(0x00, 0x4)  # Reset
        #cdma.write(0x18, int(enc_buffer.physical_address + i * PIXELS_PER_CHUNK * 4))  # Source from HP port
        #cdma.write(0x20, BRAM0_ADDR)  # Destination to BRAM
        #cdma.write(0x28, int(PIXELS_PER_CHUNK * 4))  # Total byte count
        #cdma.write(0x00, 0x1)  # Start transfer

        #while (cdma.read(0x04) & 0x2) == 0:
        #    pass

        ENC_DEC.write(0x0, 0x1)
        START.write(0x0, 0x1)

        while (DONE.read(0x0) & 0x1) == 0:
            pass

        START.write(0x0, 0x0)

        cdma.write(0x00, 0x4)  # Reset
        cdma.write(0x18, BRAM0_ADDR)  # Source from HP port
        cdma.write(0x20, int(dec_buffer.physical_address + i * PIXELS_PER_CHUNK * 4))  # Destination to BRAM
        cdma.write(0x28, int(PIXELS_PER_CHUNK * 4))  # Total byte count
        cdma.write(0x00, 0x1)  # Start transfer
        while (cdma.read(0x04) & 0x2) == 0:
            pass

    enc_data = np.array(enc_buffer).reshape((IMG_HEIGHT, IMG_WIDTH))
    #r_enc = ((enc_data >> 16) & 0xFF).astype(np.uint8)
    #g_enc = ((enc_data >> 8) & 0xFF).astype(np.uint8)
    b_enc = (enc_data & 0xFF).astype(np.uint8)

    #enc_img = np.stack([g, b, r], axis=-1).astype(np.uint8)  # shape = (720, 1280, 3)
    enc_img = cv2.merge([b_enc,b_enc,b_enc])

    dec_data = np.array(dec_buffer).reshape((IMG_HEIGHT, IMG_WIDTH))
    #r_dec = ((dec_data >> 16) & 0xFF).astype(np.uint8)
    #g_dec = ((dec_data >> 8) & 0xFF).astype(np.uint8)
    b_dec = (dec_data & 0xFF).astype(np.uint8)
    dec_img = cv2.merge([b_dec,b_dec,b_dec])
    #dec_img = np.stack([g, b, r], axis=-1).astype(np.uint8)  # shape = (720, 1280, 3)
    
    #r_o = img[:, :, 0].astype(np.uint8)
    #g_o = img[:, :, 1].astype(np.uint8)
    b_o = img[:, :, 2].astype(np.uint8)
    original_img = cv2.merge([b_o,b_o,b_o])
    #original_img = np.stack([b_o, b_o, b_o], axis=-1).astype(np.uint8)  # shape = (720, 1280, 3)

    original[:] = original_img[:]
    enc[:] = enc_img[:]
    dec[:] = dec_img[:]

    val = sw_1.read()
    val += sw_2.read()
    vdma.writechannel.writeframe(enc)
    if val == 0:
        vdma.writechannel.writeframe(original)
    elif val == 1:
        vdma.writechannel.writeframe(enc)
    else:
        vdma.writechannel.writeframe(dec)
        
    if(buttons[0].read()): 
        cap.release() 
        vdma.writechannel.stop()
        break
print("stop successfully")

[ WARN:0] global ./modules/videoio/src/cap_gstreamer.cpp (616) isPipelinePlaying OpenCV | GStreamer warning: GStreamer: pipeline have not been created


攝影機成功開啟
stop successfully
